
## 📦 Bloco 1: Importações de Bibliotecas
Dependências necessárias para o projeto rodar.


In [ ]:
from fastapi import FastAPI, HTTPException, Request, status
from fastapi.responses import JSONResponse
import httpx


### Explicação linha por linha das importações:
* `from fastapi import FastAPI, HTTPException, Request, status`:
  * `FastAPI`: É a classe principal que usamos para instanciar e gerenciar nossa aplicação web.
  * `HTTPException`: Uma classe de exceção específica do FastAPI que nos permite interromper o fluxo do código e retornar respostas de erro HTTP (ex: 400, 404) para o cliente.
  * `Request`: Objeto que representa a requisição HTTP recebida pelo servidor (usado aqui no nosso tratador de erros globais).
  * `status`: Um módulo utilitário que contém constantes para códigos de status HTTP (ex: `status.HTTP_404_NOT_FOUND`), evitando que a gente precise digitar números soltos no código.
* `from fastapi.responses import JSONResponse`: Permite gerar manualmente uma resposta formatada em JSON. É ideal para enviar respostas customizadas, como em erros globais.
* `import httpx`: Uma biblioteca moderna, rápida e que suporta programação assíncrona (`async/await`) para fazer requisições HTTP para outras APIs (neste caso, a PokeAPI externa).


## ⚙️ Bloco 2: Inicialização do App e Configuração da URL Base
Criação da aplicação e definição para onde é feita as requisições.


In [ ]:
app = FastAPI(title="POKEDEX-API")

POKEAPI_URL = "https://pokeapi.co/api/v2"


### Explicação linha por linha:
* `app = FastAPI(title="POKEDEX-API")`: Instancia o nosso servidor web. O parâmetro `title` define o nome que aparecerá automaticamente na documentação interativa gerada pelo FastAPI (como o `/docs` ou `/redoc`).
* `POKEAPI_URL = "https://pokeapi.co/api/v2"`: Define uma constante global com a URL base da API oficial do Pokémon, que usaremos como fonte de dados.


## 🚨 Bloco 3: Manipulador Global de Erros Inesperados (Erro 500)
Esse bloco impede que a  aplicação quebre visualmente caso aconteça um erro que nós não previmos no código (como uma falha de sintaxe ou erro de runtime do Python).


In [ ]:
@app.exception_handler(Exception)
async def erro_global_exception_handler(request: Request, exc: Exception):
   
    return JSONResponse(
        status_code=status.HTTP_500_INTERNAL_SERVER_ERROR,
        content={
            "erro": "Erro Interno",
            "detalhes": f"Ocorreu um erro inesperado no servidor. Tente novamente mais tarde. Erro: {str(exc)}"
        }
    )


### Explicação linha por linha:
* `@app.exception_handler(Exception)`: Um decorador do FastAPI que diz ao servidor: *"Se qualquer erro do tipo básico `Exception` (ou seja, qualquer erro genérico do Python) acontecer no sistema, capture-o e jogue para a função abaixo"*.
* `async def erro_global_exception_handler(request: Request, exc: Exception):`: Define a função assíncrona que vai tratar o erro. Ela recebe a requisição (`request`) e a exceção real que aconteceu (`exc`).
* `return JSONResponse(...)`: Retorna uma resposta estruturada em JSON contendo o status HTTP 500 (Erro Interno do Servidor).
* `content={...}`: O corpo da resposta enviado ao usuário, mascarando o erro real com uma mensagem amigável, mas incluindo o texto original da exceção convertida para string (`str(exc)`) no campo `detalhes`.


## 🏠 Bloco 4: Rota de Boas-Vindas (Endpoint Raiz)
Uma rota simples para verificar se a API está online de forma rápida.


In [ ]:
@app.get("/")
def home():
    return {"mensagem": "Bem-vindo à Minha API Pokémon! Use o endpoint /pokemon/{nome} para buscar informações sobre um Pokémon específico."}


### Explicação linha por linha:
* `@app.get("/")`: Define que quando alguém acessar a URL base do servidor por meio do método HTTP GET (ex: `http://localhost:8000/`), esta função será acionada.
* `def home():`: Cria a função síncrona chamada `home`. Como não faz requisições pesadas de rede aqui, ela não precisa ser assíncrona (`async`).
* `return {"mensagem": "..."}`: Retorna um dicionário simples que o FastAPI converte automaticamente em JSON para o navegador do cliente.


## 🔍 Bloco 5: Endpoint de Busca e Sanetização do Nome
Este é o coração da API.  analisa a assinatura da rota e a validação primária do dado digitado.


In [ ]:
@app.get("/pokemon/{nome}")
async def buscar_pokemon(nome: str):
    nome_limpo = nome.strip().lower()
    if not nome_limpo:
        raise HTTPException(
            status_code=status.HTTP_400_BAD_REQUEST, 
            detail="O nome do Pokémon não pode ser vazio."
        )


### Explicação linha por linha:
* `@app.get("/pokemon/{nome}")`: Registra uma rota do tipo GET que aceita um parâmetro dinâmico na URL chamado `{nome}` (ex: `/pokemon/pikachu`).
* `async def buscar_pokemon(nome: str):`: Cria a função assíncrona. Ela precisa ser `async` porque usará o `await` para esperar a resposta da internet mais adiante. O parâmetro `nome` é tipado estritamente como string (`str`).
* `nome_limpo = nome.strip().lower()`: 
  * `.strip()`: Remove espaços em branco acidentais antes ou depois da palavra.
  * `.lower()`: Transforma todo o texto em letras minúsculas (essencial, pois a PokeAPI não reconhece "Pikachu" com "P" maiúsculo).
* `if not nome_limpo:`: Verifica se após a limpeza, o nome ficou vazio (caso o usuário tenha digitado apenas espaços em branco).
* `raise HTTPException(...)`: Interrompe a execução imediatamente e devolve o erro **400 Bad Request** com uma mensagem avisando que o campo é obrigatório.


## 🌐 Bloco 6: Bloco Try e Requisição Assíncrona com HTTPX
Aqui o nosso servidor faz o papel de cliente, conectando-se à PokeAPI oficial para obter as informações brutas do Pokémon.


In [ ]:
    try:
      
        async with httpx.AsyncClient(timeout=5.0) as client:
            resposta = await client.get(f"{POKEAPI_URL}/pokemon/{nome_limpo}")
            
           
            resposta.raise_for_status()


### Explicação linha por linha:
* `try:`: Inicia o bloco de monitoramento. Qualquer erro de rede ou HTTP gerado aqui dentro será capturado pelos blocos `except` correspondentes abaixo.
* `async with httpx.AsyncClient(timeout=5.0) as client:`: Abre um cliente HTTP assíncrono. O gerenciador de contexto `with` garante que todas as conexões abertas sejam fechadas ao final do bloco.
  * `timeout=5.0`: Define um limite de tempo. Se a PokeAPI demorar mais que 5 segundos para nos dar um sinal de vida, a requisição é cancelada preventivamente para não travar nosso servidor.
* `resposta = await client.get(...)`: Realiza a requisição GET real de forma assíncrona. O uso de `await` sinaliza que o processador do nosso servidor pode cuidar de outras requisições enquanto espera a resposta chegar dos servidores externos.
* `resposta.raise_for_status()`: Uma função nativa muito importante. Se o código de resposta HTTP da PokeAPI for um erro (como 404, 500, etc.), ela dispara automaticamente uma exceção do tipo `httpx.HTTPStatusError`, desviando o código direto para a área de tratamento de erros.


## 🛠️ Bloco 7: Tratamento de Exceções da API Externa
Este é o bloco onde é traduzido os problemas técnicos da internet para respostas limpas e compreensíveis para o streamlit(frontend).


In [ ]:
    except httpx.HTTPStatusError as exc:
        if exc.response.status_code == 404:
            raise HTTPException(
                status_code=status.HTTP_404_NOT_FOUND, 
                detail=f"Pokémon '{nome}' não encontrado."
            )
        
        
        raise HTTPException(
            status_code=status.HTTP_502_BAD_GATEWAY,
            detail="A API externa de Pokémon apresentou instabilidade. Tente novamente mais tarde."
        )

   
    except httpx.TimeoutException:
        raise HTTPException(
            status_code=status.HTTP_504_GATEWAY_TIMEOUT,
            detail="O servidor externo demorou muito para responder. Limite de tempo esgotado."
        )
    except httpx.RequestError:
        raise HTTPException(
            status_code=status.HTTP_503_SERVICE_UNAVAILABLE,
            detail="Não foi possível estabelecer conexão com o serviço externo de Pokémon."
        )


### Explicação linha por linha:
* `except httpx.HTTPStatusError as exc:`: Captura falhas geradas pelo `raise_for_status()`.
* `if exc.response.status_code == 404:`: Se o status retornado pela PokeAPI foi 404, significa que o nome digitado não corresponde a nenhum Pokémon.
* `raise HTTPException(status_code=status.HTTP_404_NOT_FOUND, ...)`: Repassa o erro **404 Not Found** personalizado para o nosso frontend saber exatamente que o Pokémon não existe.
* `raise HTTPException(status_code=status.HTTP_502_BAD_GATEWAY, ...)`: Se o erro HTTP não foi 404, mas sim outro (como um erro 500 interno deles), assumimos que a PokeAPI está instável e retornamos um erro **502 Bad Gateway** (Portão de Ligação Inválido).
* `except httpx.TimeoutException:`: Captura especificamente o estouro dos 5 segundos que estipulamos no cliente HTTPX.
* `raise HTTPException(status_code=status.HTTP_504_GATEWAY_TIMEOUT, ...)`: Retorna um erro **504 Gateway Timeout**, explicando o estouro de tempo.
* `except httpx.RequestError:`: Captura falhas físicas de rede (ex: o servidor da API externa caiu por completo, falha de DNS ou falta de internet na máquina host).
* `raise HTTPException(status_code=status.HTTP_503_SERVICE_UNAVAILABLE, ...)`: Retorna um erro **503 Service Unavailable**, indicando indisponibilidade total do ecossistema externo.


## 📦 Bloco 8: Formatação e Tratamento do JSON Resultante
Uma vez coletados os dados brutos da PokeAPI com sucesso, é filtrado apenas o que interessa, tratar as unidades de medida e blindar contra falhas de chaves ausentes.


In [ ]:
    try:
        dados = resposta.json()

        return {
            "id": dados["id"],
            "nome": dados["name"],
            "altura": dados["height"] / 10,
            "peso": dados["weight"] / 10,
            "tipos": [tipo["type"]["name"] for tipo in dados["types"]],
            "habilidades": [
                habilidade["ability"]["name"]
                for habilidade in dados["abilities"]
            ],
            "imagem": dados.get("sprites", {})
                           .get("other", {})
                           .get("official-artwork", {})
                           .get("front_default")
        }
    except (ValueError, KeyError, TypeError):
        raise HTTPException(
            status_code=status.HTTP_502_BAD_GATEWAY,
            detail="A estrutura de dados retornada pelo servidor externo é inválida ou incompatível."
        )


### Explicação linha por linha:
* `try:`: Inicia um novo bloco de proteção focado exclusivamente na leitura dos dados mapeados.
* `dados = resposta.json()`: Desconverte a string bruta recebida e a transforma em um dicionário Python maleável.
* `return { ... }`: Monta o JSON de resposta final limpo que o nosso app Streamlit espera receber.
  * `"id": dados["id"]`: Extrai o número identificador do Pokémon.
  * `"nome": dados["name"]`: Extrai o nome textual.
  * `"altura": dados["height"] / 10`: **Correção de Unidade!** A PokeAPI retorna a altura em *decímetros*. Dividindo por 10, convertemos corretamente para *metros*.
  * `"peso": dados["weight"] / 10`: **Correção de Unidade!** A PokeAPI retorna o peso em *hectogramas*. Dividindo por 10, convertemos corretamente para *quilogramas (kg)*.
  * `"tipos": [tipo["type"]["name"] for tipo in dados["types"]]`: Uma *List Comprehension* que navega pela lista complexa de tipos e extrai apenas o nome de cada um (ex: `['electric']`).
  * `"habilidades": [...]`: Outra *List Comprehension* que extrai unicamente o texto bruto dos nomes das habilidades disponíveis para aquele espécime.
  * `"imagem": dados.get("sprites", {})...get("front_default")`: Usa múltiplos encadeamentos do método `.get()`. Caso a PokeAPI pare de enviar fotos ou mude o nome de alguma subpasta interna, o código retornará `None` de forma segura em vez de disparar uma quebra abrupta de dicionário.
* `except (ValueError, KeyError, TypeError):`: Captura falhas estruturais (se o JSON veio corrompido, se alguma das chaves acima não existe mais ou se tentamos dividir um dado nulo por 10).
* `raise HTTPException(status_code=status.HTTP_502_BAD_GATEWAY, ...)`: Retorna um **502 Bad Gateway** explicando que os dados externos são incompatíveis, mantendo a robustez e elegância do ecossistema.
